In [2]:
import ipycanvas
from ipywidgets import IntSlider
slider = IntSlider(value=5, min=0, max=10)
slider


IntSlider(value=5, max=10)

In [3]:
from ipycanvas import Canvas
canvas = Canvas(width=200, height=200)
canvas.stroke_rect(10, 10, 180, 180)
canvas


Canvas(height=200, width=200)

In [1]:
import numpy as np
from ipycanvas import Canvas
from ipywidgets import VBox, HBox, Button, IntText, Output, Label
import matplotlib.pyplot as plt

# Canvas (560x560 für 16x16)
canvas = Canvas(width=560, height=560)

# Zeichnen-Setup
canvas.fill_style = 'white'
canvas.fill_rect(0, 0, 560, 560)
canvas.stroke_style = '#333'
canvas.line_width = 35
canvas.line_cap = 'round'
canvas.line_join = 'round'

# Globale Variablen
drawing = False
x_prev, y_prev = 0, 0
my_dataset = []  # Deine Daten: [{'features': array, 'label': 0}, ...]

# Zeichnen-Funktionen
def on_mouse_down(x, y):
    global drawing, x_prev, y_prev
    drawing = True
    x_prev, y_prev = x, y

def on_mouse_move(x, y):
    global x_prev, y_prev
    if drawing:
        canvas.stroke_style = 'black'
        canvas.line_width = 35
        canvas.stroke_line(x_prev, y_prev, x, y)
        x_prev, y_prev = x, y

def on_mouse_up(x, y):
    global drawing
    drawing = False

canvas.on_mouse_down(on_mouse_down)
canvas.on_mouse_move(on_mouse_move)
canvas.on_mouse_up(on_mouse_up)

# 16x16 → NumPy (0-16 wie sklearn)
def canvas_to_16x16(label):
    """Canvas → 16x16 Array (0-16)"""
    GRID_SIZE = 16
    cell_w, cell_h = 560 / GRID_SIZE, 560 / GRID_SIZE
    
    # ImageData
    image_data = canvas.get_image_data(0, 0, 560, 560)
    
    features = np.zeros(GRID_SIZE * GRID_SIZE)
    
    for gy in range(GRID_SIZE):
        for gx in range(GRID_SIZE):
            start_x = int(gx * cell_w)
            start_y = int(gy * cell_h)
            end_x = int((gx + 1) * cell_w)
            end_y = int((gy + 1) * cell_h)
            
            sum_intensity = 0
            count = 0
            
            for py in range(start_y, end_y):
                for px in start_x, end_x:
                    idx = (py * 560 + px) * 4
                    r, g, b = image_data.data[idx:idx+3]
                    gray = (r + g + b) / (3 * 255)
                    intensity = 1 - gray  # Schwärze
                    sum_intensity += intensity
                    count += 1
            
            if count > 0:
                features[gy*GRID_SIZE + gx] = round(sum_intensity / count * 16)
    
    data_point = {'features': features, 'label': label}
    my_dataset.append(data_point)
    return data_point

# Widgets
label_input = IntText(value=0, min=0, max=9, description='Label:')
clear_btn = Button(description='🗑️ Leeren', button_style='warning')
add_btn = Button(description='➕ 16x16 speichern', button_style='success')
show_btn = Button(description='📊 Dataset zeigen', button_style='info')

out = Output()
status = Label(value='Zeichne deine Ziffer!')

def clear(b):
    global drawing
    canvas.fill_style = 'white'
    canvas.fill_rect(0, 0, 560, 560)
    canvas.stroke_style = '#333'
    canvas.stroke_rect(0, 0, 560, 560)
    status.value = 'Canvas geleert'

def add_data(b):
    label = label_input.value
    data = canvas_to_16x16(label)
    status.value = f'✅ Gespeichert: Label {label} | Dataset: {len(my_dataset)} Zeilen'
    print(f"Neue Zeile: Label={label}, Shape={data['features'].shape}")
    print(f"Erste 10 Werte: {data['features'][:10]}")

def show_dataset(b):
    with out:
        out.clear_output()
        if my_dataset:
            print(f"\n🎯 DATASET ({len(my_dataset)} Zeilen):")
            features = np.array([d['features'] for d in my_dataset])
            labels = np.array([d['label'] for d in my_dataset])
            
            print(f"Shape: {features.shape}")
            print(f"Labels: {labels}")
            print("\nLetzte Zeile:")
            print(features[-1].reshape(16,16))
            
            # Plot
            fig, axes = plt.subplots(1, 3, figsize=(15,4))
            axes[0].imshow(features[-1].reshape(16,16), cmap='gray')
            axes[0].set_title('Letzte Zeichnung')
            axes[1].hist(features.flatten(), bins=17)
            axes[1].set_title('Werte-Verteilung (0-16)')
            axes[2].bar(range(10), np.bincount(labels, minlength=10))
            axes[2].set_title('Zeilen pro Label')
            plt.tight_layout()
            plt.show()
        else:
            print("❌ Noch keine Daten!")

clear_btn.on_click(clear)
add_btn.on_click(add_data)
show_btn.on_click(show_dataset)

# UI
display(VBox([
    HBox([canvas, VBox([label_input, clear_btn, add_btn])]),
    status,
    HBox([show_btn]),
    out
]))


RuntimeError: No image data, please be sure that ``sync_image_data`` is set to True